# FOXF1_BMP4_timelapse — 03c_segmentation_ilastik_prob_tracking

**Feeds:** Fig 5i, ED Fig 10f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# FOXF1/BMP4 Timelapse: Step 3c Ilastik-Driven Segmentation + Division-Aware Tracking

This notebook uses ilastik pixel-class probabilities as the primary signal for nucleus segmentation.
It is intentionally staged:

1. **Fast integrity + visual checks** (seconds to a few minutes)
2. **Fast search (stage 1)** on sampled frames (moderate runtime)
3. **Optional long search (stage 2)** with temporal/division metrics (longer runtime)
4. **Final export** of overlays, tracks, and QC tables

The goal is to avoid long blind runs and provide interpretable intermediate outputs.


In [ ]:
import json
import math
import time
from dataclasses import dataclass
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from IPython.display import display

from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

from skimage import filters, feature, morphology, measure, segmentation, exposure, color

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
np.set_printoptions(suppress=True, precision=4)


In [ ]:
# ----------------------------- #
# Global configuration
# ----------------------------- #
def find_project_root(start: Path) -> Path:
    """Find repo root by locating ilastik/datasets/20260213/batch_processing_data_v0 in cwd or parents."""
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if (cand / "ilastik" / "datasets" / "20260213" / "batch_processing_data_v0").exists():
            return cand
    return start


ROOT = find_project_root(Path.cwd())
ILASTIK_DIR = ROOT / "ilastik" / "datasets" / "20260213" / "batch_processing_data_v0"
RAW_PATH = ILASTIK_DIR / "1-Pos009_012.tif"
PROB_PATH = ILASTIK_DIR / "1-Pos009_012_Probabilities.tiff"

RESULTS_DIR = ROOT / "results" / "datasets" / "20260213" / "segmentation_03c_ilastik"
QUICK_DIR = RESULTS_DIR / "quick_checks"
SEARCH_DIR = RESULTS_DIR / "search"
EXPORT_DIR = RESULTS_DIR / "exports"
for _d in [RESULTS_DIR, QUICK_DIR, SEARCH_DIR, EXPORT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

POSITION_LABEL = "1-Pos009_012"
MIN_PER_FRAME = 10.0

# Top-left exclusion region strategy:
# - auto estimate from low-illumination area in time-median RFP,
# - then optionally union with a manual triangular wedge.
USE_AUTO_EXCLUSION_MASK = True
AUTO_MASK_TIME_STRIDE = 5
AUTO_MASK_GAUSS_SIGMA = 35
AUTO_MASK_LOW_PERCENTILE = 7.0
AUTO_MASK_CORNER_PATCH = 40
AUTO_MASK_DILATE = 3
UNION_MANUAL_WEDGE = True
MANUAL_WEDGE_SLOPE = -1.10
MANUAL_WEDGE_INTERCEPT = 115.0   # y < m*x + b is excluded

# Preview frames for quick visual checks.
PREVIEW_TIMES = [0, 20, 40, 80, 120, 160, 220, 280, 299]
# Three representative timepoints for step-by-step segmentation diagnostics.
# If None, defaults to [0, N_TIME // 2, N_TIME - 1].
STEP_DIAG_TIMES = None

# Stage-1 search (fast): framewise metrics only on sampled frames.
RUN_STAGE1_SEARCH = True
STAGE1_N_CANDIDATES = 48
STAGE1_SAMPLE_N_FRAMES = 96
STAGE1_RANDOM_SEED = 13

# Stage-2 search (slower): segmentation + tracking metrics.
# Keep this optional. Only run after inspecting stage-1 outputs.
RUN_STAGE2_SEARCH = False
STAGE2_TOPK_FROM_STAGE1 = 8
STAGE2_N_REFINEMENTS_PER_BASE = 4
STAGE2_RANDOM_SEED = 31
STAGE2_PERSISTENCE_K = 3
STAGE2_EVAL_MAX_FRAMES = None  # set int for faster dry-runs

# Final export behavior.
FULL_RUN_MAX_FRAMES = None  # set int for faster dry-runs
EXPORT_FULL_OVERLAY_STACK = True
EXPORT_MONTAGE = True
MONTAGE_COLS = 20


## 1) Load Data And Verify Ilastik Encoding

We expect:

- Raw TIFF pages: alternating `(BF, RFP)` => `(0,1), (2,3), ...`
- Probability TIFF pages: alternating `(p_nuclei, p_non_nuclei)` => `(0,1), (2,3), ...`
- Probability pairs should satisfy `p_nuc + p_non_nuc == 1` (within floating tolerance)


In [ ]:
assert RAW_PATH.exists(), f"Missing raw input: {RAW_PATH}"
assert PROB_PATH.exists(), f"Missing probability input: {PROB_PATH}"

with tifffile.TiffFile(RAW_PATH) as tf_raw:
    n_pages_raw = len(tf_raw.pages)
    shape_raw = tf_raw.pages[0].shape
    dtype_raw = tf_raw.pages[0].dtype

with tifffile.TiffFile(PROB_PATH) as tf_prob:
    n_pages_prob = len(tf_prob.pages)
    shape_prob = tf_prob.pages[0].shape
    dtype_prob = tf_prob.pages[0].dtype

print(f"RAW:  pages={n_pages_raw}, shape={shape_raw}, dtype={dtype_raw}")
print(f"PROB: pages={n_pages_prob}, shape={shape_prob}, dtype={dtype_prob}")

assert n_pages_raw % 2 == 0, "Raw TIFF must have even page count (BF/RFP pairs)."
assert n_pages_prob % 2 == 0, "Probability TIFF must have even page count (nuc/non-nuc pairs)."
assert n_pages_raw == n_pages_prob, "Raw/probability page counts should match in this export."
assert shape_raw == shape_prob, "Raw/probability XY shape mismatch."


In [ ]:
# Memory-map both stacks so we can access frames without loading all pages into RAM.
raw_mm = tifffile.memmap(RAW_PATH)
prob_mm = tifffile.memmap(PROB_PATH)
print("raw memmap shape:", raw_mm.shape, "dtype:", raw_mm.dtype)
print("prob memmap shape:", prob_mm.shape, "dtype:", prob_mm.dtype)

assert raw_mm.ndim == 4 and raw_mm.shape[1] == 2, "Expected raw memmap shape (T,2,Y,X)."
assert prob_mm.ndim == 4 and prob_mm.shape[1] == 2, "Expected prob memmap shape (T,2,Y,X)."
assert raw_mm.shape[0] == prob_mm.shape[0], "Raw/prob memmap time dimension mismatch."

N_TIME = int(raw_mm.shape[0])
print(f"Derived timepoints from memmap: {N_TIME}")
assert n_pages_raw == 2 * N_TIME, "Page count and memmap time dim mismatch for raw TIFF."
assert n_pages_prob == 2 * N_TIME, "Page count and memmap time dim mismatch for prob TIFF."

# Quick empirical checks for alternating structure and probability complementarity.
sample_times = [0, 1, 20, 50, 100, 150, 200, 250, N_TIME - 1]
sample_times = sorted({i for i in sample_times if 0 <= i < N_TIME})

pair_checks = []
for t in sample_times:
    p0 = prob_mm[t, 0].astype(np.float32, copy=False)
    p1 = prob_mm[t, 1].astype(np.float32, copy=False)
    s = p0 + p1
    pair_checks.append(
        {
            "pair_start_page": int(2 * t),
            "pair_index": int(t),
            "p0_mean": float(p0.mean()),
            "p1_mean": float(p1.mean()),
            "sum_mean": float(s.mean()),
            "sum_std": float(s.std()),
            "sum_min": float(s.min()),
            "sum_max": float(s.max()),
            "max_abs_err_from_1": float(np.max(np.abs(s - 1.0))),
        }
    )

pair_checks_df = pd.DataFrame(pair_checks)
display(pair_checks_df)

# Confirm the expected representation: first page of each pair is nuclei probability.
# In memmap form this corresponds to channel index 0 at each time.


In [ ]:
# Build a frame table for easy downstream indexing and plotting.
frame_df = pd.DataFrame(
    {
        "time": np.arange(N_TIME, dtype=int),
        "hours": np.arange(N_TIME, dtype=float) * (MIN_PER_FRAME / 60.0),
        "bf_page": np.arange(0, n_pages_raw, 2, dtype=int),
        "rfp_page": np.arange(1, n_pages_raw, 2, dtype=int),
        "pnuc_page": np.arange(0, n_pages_prob, 2, dtype=int),
        "pbg_page": np.arange(1, n_pages_prob, 2, dtype=int),
        "bf_ch": np.zeros(N_TIME, dtype=int),
        "rfp_ch": np.ones(N_TIME, dtype=int),
        "pnuc_ch": np.zeros(N_TIME, dtype=int),
        "pbg_ch": np.ones(N_TIME, dtype=int),
        "position_label": POSITION_LABEL,
    }
)
display(frame_df.head())
display(frame_df.tail())


In [ ]:
def get_raw_prob_frame(time_idx: int):
    """Load one timepoint as BF, RFP, p_nuc, p_non_nuc float arrays."""
    row = frame_df.iloc[int(time_idx)]
    bf = raw_mm[int(row.time), int(row.bf_ch)].astype(np.float32)
    rfp = raw_mm[int(row.time), int(row.rfp_ch)].astype(np.float32)
    p_nuc = prob_mm[int(row.time), int(row.pnuc_ch)].astype(np.float32)
    p_non = prob_mm[int(row.time), int(row.pbg_ch)].astype(np.float32)
    return bf, rfp, p_nuc, p_non


n_show = min(len(PREVIEW_TIMES), 9)
fig, axes = plt.subplots(n_show, 4, figsize=(12, 2.6 * n_show), constrained_layout=True)
if n_show == 1:
    axes = axes[None, :]

for r, t in enumerate(PREVIEW_TIMES[:n_show]):
    bf, rfp, p_nuc, p_non = get_raw_prob_frame(int(t))
    s = p_nuc + p_non
    ax = axes[r]
    ax[0].imshow(bf, cmap="gray")
    ax[0].set_title(f"BF raw t={t}")
    ax[1].imshow(rfp, cmap="gray")
    ax[1].set_title("RFP raw")
    ax[2].imshow(p_nuc, cmap="magma", vmin=0, vmax=1)
    ax[2].set_title("p_nuc (class 0)")
    ax[3].imshow(s, cmap="viridis", vmin=0.999, vmax=1.001)
    ax[3].set_title("p_nuc + p_non")
    for a in ax:
        a.set_xticks([])
        a.set_yticks([])
plt.show()


## 2) Build Exclusion Mask (Top-Left Artifact Region)

We estimate a stable top-left low-illumination region from time-median RFP and exclude it from all segmentation/tracking metrics.


In [ ]:
def estimate_top_left_exclusion_mask() -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    """Estimate top-left exclusion area from low-illumination RFP in time-median image."""
    sample_times = np.arange(0, N_TIME, AUTO_MASK_TIME_STRIDE, dtype=int)
    rfp_stack = np.stack([raw_mm[int(t), 1].astype(np.float32) for t in sample_times], axis=0)
    rfp_med = np.median(rfp_stack, axis=0)

    # Heavy blur captures large-scale illumination instead of cellular texture.
    illum = filters.gaussian(rfp_med, sigma=AUTO_MASK_GAUSS_SIGMA, preserve_range=True)
    thr = float(np.percentile(illum, AUTO_MASK_LOW_PERCENTILE))
    low = illum <= thr

    # Keep only connected low region touching the top-left patch.
    low = morphology.binary_closing(low, footprint=morphology.disk(7))
    low = morphology.binary_opening(low, footprint=morphology.disk(2))
    lbl = measure.label(low)
    corner = lbl[:AUTO_MASK_CORNER_PATCH, :AUTO_MASK_CORNER_PATCH]
    touch = np.unique(corner)
    touch = touch[touch > 0]
    keep = np.isin(lbl, touch)
    if AUTO_MASK_DILATE > 0:
        keep = morphology.binary_dilation(keep, footprint=morphology.disk(AUTO_MASK_DILATE))

    if UNION_MANUAL_WEDGE:
        yy, xx = np.indices(keep.shape)
        wedge = yy < (MANUAL_WEDGE_SLOPE * xx + MANUAL_WEDGE_INTERCEPT)
        keep = keep | wedge

    return keep.astype(bool), rfp_med, illum.astype(np.float32), thr


if USE_AUTO_EXCLUSION_MASK:
    exclusion_mask, rfp_time_median, illum_map, illum_thr = estimate_top_left_exclusion_mask()
else:
    exclusion_mask = np.zeros(shape_raw, dtype=bool)
    rfp_time_median = raw_mm[1].astype(np.float32)
    illum_map = rfp_time_median.copy()
    illum_thr = np.nan

valid_mask = ~exclusion_mask
print(f"Excluded pixels: {exclusion_mask.mean() * 100:.2f}%")
print(f"Valid pixels: {valid_mask.mean() * 100:.2f}%")

fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
axes[0].imshow(rfp_time_median, cmap="gray")
axes[0].set_title("Time-median RFP")
axes[1].imshow(illum_map, cmap="magma")
axes[1].set_title(f"Smoothed illumination\n(thr={illum_thr:.2f})")
axes[2].imshow(exclusion_mask, cmap="gray")
axes[2].set_title("Exclusion mask")

overlay = np.dstack([exposure.rescale_intensity(rfp_time_median, out_range=(0, 1))] * 3)
contour = segmentation.find_boundaries(exclusion_mask, mode="outer")
overlay[contour] = np.array([0.0, 1.0, 1.0], dtype=np.float32)
axes[3].imshow(overlay)
axes[3].set_title("Mask boundary on RFP")
for a in axes:
    a.set_xticks([])
    a.set_yticks([])
plt.show()


## 3) Segmentation Functions (Probability-Primary)

Key idea: segmentation starts from `p_nuc`, not raw intensity thresholds.


In [ ]:
DEFAULT_SEG_PARAMS = {
    # Smoothing for probability map before threshold/seed extraction.
    "smooth_sigma": 1.1,
    # Hysteresis-like region support.
    "low_prob_thr": 0.30,
    "high_prob_thr": 0.55,
    # Seed settings for watershed splitting.
    "seed_prob_thr": 0.44,
    "seed_min_distance": 6,
    "max_seed_peaks": 1200,
    # Morphology and object filtering.
    "open_radius": 1,
    "close_radius": 2,
    "min_hole_area": 64,
    "min_area": 65,
    "max_area": 2200,
    "min_solidity": 0.55,
    "max_eccentricity": 0.995,
    "min_mean_prob": 0.36,
    # Metrics cutoffs.
    "high_recall_prob_thr": 0.80,
    "low_contam_prob_thr": 0.20,
    "small_obj_area_cutoff": 90,
    "large_obj_area_cutoff": 700,
    # Shape quality cutoffs used by objective (not hard segmentation gates).
    "shape_low_solidity_thr": 0.85,
    "shape_low_circularity_thr": 0.80,
    "shape_high_ecc_thr": 0.90,
    # Hard split-control (within-frame) to reduce over-segmentation.
    "split_merge_enabled": True,
    "split_merge_max_passes": 2,
    "split_merge_boundary_prob_thr": 0.62,
    "split_merge_min_contact_px": 8,
    "split_merge_max_centroid_dist": 11.0,
    "split_merge_small_area": 150,
    "split_merge_shape_gain_min": 0.08,
}


def clamp_probs(p: np.ndarray) -> np.ndarray:
    return np.clip(p.astype(np.float32, copy=False), 0.0, 1.0)


def segment_probability_frame(
    p_nuc: np.ndarray,
    valid_mask: np.ndarray,
    params: dict,
    return_debug: bool = False,
) -> dict:
    """Segment nuclei instances from nuclei-probability image."""
    p = clamp_probs(p_nuc)
    p_sm = filters.gaussian(p, sigma=float(params["smooth_sigma"]), preserve_range=True)
    p_sm = np.where(valid_mask, p_sm, 0.0)
    debug_steps = {}
    if return_debug:
        debug_steps["p_raw"] = p.copy()
        debug_steps["p_smooth"] = p_sm.copy()

    low = (p_sm >= float(params["low_prob_thr"])) & valid_mask
    high = (p_sm >= float(params["high_prob_thr"])) & valid_mask

    # Hysteresis-like support: only keep low-threshold regions connected to high-confidence core.
    hys = ndi.binary_propagation(high, mask=low)
    if return_debug:
        debug_steps["mask_low"] = low.copy()
        debug_steps["mask_high"] = high.copy()
        debug_steps["hys_pre_fallback"] = hys.copy()

    # If high threshold is too strict in some frames, allow fallback to low regions.
    if not np.any(hys):
        hys = low.copy()
    if return_debug:
        debug_steps["hys_after_fallback"] = hys.copy()

    if int(params["open_radius"]) > 0:
        hys = morphology.binary_opening(hys, footprint=morphology.disk(int(params["open_radius"])))
    if int(params["close_radius"]) > 0:
        hys = morphology.binary_closing(hys, footprint=morphology.disk(int(params["close_radius"])))

    hys = morphology.remove_small_holes(hys, area_threshold=int(params["min_hole_area"]))
    hys = morphology.remove_small_objects(hys, min_size=max(8, int(params["min_area"] // 2)))
    hys = hys & valid_mask
    if return_debug:
        debug_steps["hys_post_morph"] = hys.copy()

    # Create watershed markers from probability peaks inside candidate mask.
    peak_coords = feature.peak_local_max(
        p_sm,
        labels=hys.astype(np.uint8),
        min_distance=int(params["seed_min_distance"]),
        threshold_abs=float(params["seed_prob_thr"]),
        num_peaks=int(params["max_seed_peaks"]),
        exclude_border=False,
    )
    markers = np.zeros_like(hys, dtype=np.int32)
    if peak_coords.size > 0:
        markers[peak_coords[:, 0], peak_coords[:, 1]] = np.arange(1, peak_coords.shape[0] + 1, dtype=np.int32)
        markers = ndi.label(markers > 0)[0].astype(np.int32)
    if return_debug:
        debug_steps["markers_peak"] = markers.copy()

    # Ensure each connected component has at least one marker.
    if markers.max() == 0:
        markers = ndi.label(hys)[0].astype(np.int32)
    else:
        cc = ndi.label(hys)[0]
        for cc_id in range(1, int(cc.max()) + 1):
            m = cc == cc_id
            if not np.any(markers[m] > 0):
                rr, ccx = np.where(m)
                if rr.size > 0:
                    # Place fallback marker at highest-probability pixel in this component.
                    idx = np.argmax(p_sm[rr, ccx])
                    markers[rr[idx], ccx[idx]] = markers.max() + 1
        markers = ndi.label(markers > 0)[0].astype(np.int32)
    if return_debug:
        debug_steps["markers_final"] = markers.copy()

    labels = segmentation.watershed(-p_sm, markers=markers, mask=hys).astype(np.int32)
    if return_debug:
        debug_steps["labels_watershed"] = labels.copy()

    # ------------------------------------------------------------------ #
    # Hard split-control:
    # Merge adjacent watershed fragments when boundary/shape evidence
    # suggests they are pieces of one nucleus rather than two cells.
    # ------------------------------------------------------------------ #
    def _shape_stats(lbl_arr: np.ndarray) -> pd.DataFrame:
        if int(lbl_arr.max()) == 0:
            return pd.DataFrame(
                columns=[
                    "label",
                    "area",
                    "solidity",
                    "eccentricity",
                    "centroid-0",
                    "centroid-1",
                    "perimeter",
                    "mean_intensity",
                    "max_intensity",
                    "circularity",
                ]
            )
        ptab = measure.regionprops_table(
            lbl_arr,
            intensity_image=p_sm,
            properties=(
                "label",
                "area",
                "solidity",
                "eccentricity",
                "centroid",
                "perimeter",
                "mean_intensity",
                "max_intensity",
            ),
        )
        df = pd.DataFrame(ptab)
        if df.empty:
            return df
        per = df["perimeter"].to_numpy(dtype=float)
        ar = df["area"].to_numpy(dtype=float)
        circ = np.where(per > 0, 4.0 * np.pi * ar / (per ** 2), np.nan)
        df["circularity"] = circ
        return df

    def _adjacency_boundary_means(lbl_arr: np.ndarray) -> dict:
        if int(lbl_arr.max()) <= 1:
            return {}
        base = int(lbl_arr.max()) + 1
        pid_parts = []
        val_parts = []

        # Horizontal and vertical label adjacency with average boundary probability.
        for la, lb, va, vb in [
            (lbl_arr[:, :-1], lbl_arr[:, 1:], p_sm[:, :-1], p_sm[:, 1:]),
            (lbl_arr[:-1, :], lbl_arr[1:, :], p_sm[:-1, :], p_sm[1:, :]),
        ]:
            m = (la > 0) & (lb > 0) & (la != lb)
            if not np.any(m):
                continue
            a = la[m].astype(np.int32)
            b = lb[m].astype(np.int32)
            lo = np.minimum(a, b)
            hi2 = np.maximum(a, b)
            pid = lo.astype(np.int64) * base + hi2.astype(np.int64)
            vv = 0.5 * (va[m].astype(np.float32) + vb[m].astype(np.float32))
            pid_parts.append(pid)
            val_parts.append(vv)

        if not pid_parts:
            return {}

        pid_all = np.concatenate(pid_parts, axis=0)
        val_all = np.concatenate(val_parts, axis=0).astype(np.float64)
        uniq, inv = np.unique(pid_all, return_inverse=True)
        sums = np.bincount(inv, weights=val_all)
        cnts = np.bincount(inv)

        out = {}
        for u, s, c in zip(uniq, sums, cnts):
            a = int(u // base)
            b = int(u % base)
            out[(a, b)] = {
                "boundary_mean_prob": float(s / max(c, 1)),
                "contact_px": int(c),
            }
        return out

    if bool(params.get("split_merge_enabled", True)) and int(labels.max()) > 1:
        max_passes = int(params.get("split_merge_max_passes", 0))
        for _ in range(max_passes):
            stats_df = _shape_stats(labels)
            if stats_df.empty or int(labels.max()) <= 1:
                break
            stats_df = stats_df.set_index("label")
            pair_info = _adjacency_boundary_means(labels)
            if not pair_info:
                break

            best_pair = None
            best_priority = -np.inf

            for (a, b), info in pair_info.items():
                if (a not in stats_df.index) or (b not in stats_df.index):
                    continue

                contact_px = int(info["contact_px"])
                if contact_px < int(params["split_merge_min_contact_px"]):
                    continue

                boundary_mean = float(info["boundary_mean_prob"])
                if boundary_mean < float(params["split_merge_boundary_prob_thr"]):
                    continue

                ra = stats_df.loc[a]
                rb = stats_df.loc[b]
                cy_a, cx_a = float(ra["centroid-0"]), float(ra["centroid-1"])
                cy_b, cx_b = float(rb["centroid-0"]), float(rb["centroid-1"])
                dist = float(np.hypot(cy_a - cy_b, cx_a - cx_b))
                if dist > float(params["split_merge_max_centroid_dist"]):
                    continue

                area_a = float(ra["area"])
                area_b = float(rb["area"])
                area_ratio = max(area_a, area_b) / max(min(area_a, area_b), 1e-6)

                merged_mask = (labels == int(a)) | (labels == int(b))
                mprops = measure.regionprops(merged_mask.astype(np.uint8))
                if len(mprops) == 0:
                    continue
                mr = mprops[0]
                m_sol = float(mr.solidity) if mr.solidity is not None else np.nan
                m_per = float(mr.perimeter) if mr.perimeter is not None else np.nan
                m_circ = (4.0 * np.pi * float(mr.area) / (m_per ** 2)) if (np.isfinite(m_per) and m_per > 0) else np.nan
                if (not np.isfinite(m_sol)) or (not np.isfinite(m_circ)):
                    continue

                sa = float(ra["solidity"]) if np.isfinite(ra["solidity"]) else 0.0
                sb = float(rb["solidity"]) if np.isfinite(rb["solidity"]) else 0.0
                ca = float(ra["circularity"]) if np.isfinite(ra["circularity"]) else 0.0
                cb = float(rb["circularity"]) if np.isfinite(rb["circularity"]) else 0.0
                before_shape = max(sa + ca, sb + cb)
                after_shape = m_sol + m_circ
                gain = float(after_shape - before_shape)

                small_cond = min(area_a, area_b) <= float(params["split_merge_small_area"])
                shape_cond = gain >= float(params["split_merge_shape_gain_min"])
                # Guard: avoid collapsing likely true neighboring nuclei unless shape gain is strong.
                if (not small_cond) and (area_ratio < 1.7) and (
                    gain < float(params["split_merge_shape_gain_min"]) + 0.10
                ):
                    continue
                if not (small_cond or shape_cond):
                    continue

                priority = (
                    gain
                    + 0.35 * (boundary_mean - float(params["split_merge_boundary_prob_thr"]))
                    + 0.002 * float(contact_px)
                )
                if priority > best_priority:
                    best_priority = priority
                    best_pair = (int(a), int(b))

            if best_pair is None:
                break

            a, b = best_pair
            labels[labels == b] = a
            labels, _, _ = segmentation.relabel_sequential(labels)
    if return_debug:
        debug_steps["labels_splitmerge"] = labels.copy()

    # Post-filter regions by morphology/probability.
    props = measure.regionprops(labels, intensity_image=p_sm)
    keep = []
    rows = []
    for r in props:
        area = float(r.area)
        solidity = float(r.solidity) if r.solidity is not None else np.nan
        ecc = float(r.eccentricity) if r.eccentricity is not None else np.nan
        mean_p = float(r.mean_intensity)
        max_p = float(r.max_intensity)
        perimeter = float(r.perimeter) if r.perimeter is not None else np.nan
        circ = (4.0 * np.pi * area / (perimeter ** 2)) if (perimeter is not None and perimeter > 0) else np.nan

        ok = True
        if area < float(params["min_area"]) or area > float(params["max_area"]):
            ok = False
        if np.isfinite(solidity) and solidity < float(params["min_solidity"]):
            ok = False
        if np.isfinite(ecc) and ecc > float(params["max_eccentricity"]):
            ok = False
        if mean_p < float(params["min_mean_prob"]):
            ok = False

        rows.append(
            {
                "label": int(r.label),
                "area": area,
                "solidity": solidity,
                "eccentricity": ecc,
                "mean_prob": mean_p,
                "max_prob": max_p,
                "circularity": circ,
                "centroid_y": float(r.centroid[0]),
                "centroid_x": float(r.centroid[1]),
                "keep": bool(ok),
            }
        )
        if ok:
            keep.append(int(r.label))

    if keep:
        labels = np.where(np.isin(labels, np.array(keep, dtype=np.int32)), labels, 0)
        labels, _, _ = segmentation.relabel_sequential(labels)
    else:
        labels = np.zeros_like(labels, dtype=np.int32)
    if return_debug:
        debug_steps["labels_final"] = labels.copy()

    regions_df = pd.DataFrame(rows)
    if not regions_df.empty:
        # Recompute label IDs after relabeling.
        kept = regions_df[regions_df["keep"]].copy().reset_index(drop=True)
    else:
        kept = pd.DataFrame(
            columns=[
                "label",
                "area",
                "solidity",
                "eccentricity",
                "mean_prob",
                "max_prob",
                "circularity",
                "centroid_y",
                "centroid_x",
                "keep",
            ]
        )

    out = {
        "p_smooth": p_sm.astype(np.float32),
        "binary": hys.astype(bool),
        "labels": labels.astype(np.int32),
        "markers": markers.astype(np.int32),
        "regions_df": kept,
        "peak_coords": peak_coords,
    }
    if return_debug:
        out["debug_steps"] = debug_steps
    return out


def _robust_gray_display(
    gray: np.ndarray,
    low_pct: float = 1.0,
    high_pct: float = 99.5,
) -> np.ndarray:
    """Robust display scaling that ignores masked corner artifacts."""
    g = gray.astype(np.float32)
    vm = globals().get("valid_mask", None)
    vals = g[vm] if (vm is not None and vm.shape == g.shape) else g.reshape(-1)
    if vals.size == 0:
        lo, hi = float(np.min(g)), float(np.max(g))
    else:
        lo = float(np.percentile(vals, low_pct))
        hi = float(np.percentile(vals, high_pct))
    if hi <= lo:
        hi = lo + 1.0
    out = np.clip((g - lo) / (hi - lo), 0.0, 1.0)
    if vm is not None and vm.shape == g.shape:
        out = out.copy()
        out[~vm] = 0.0
    return out


def overlay_labels_on_gray(gray: np.ndarray, labels: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """Color-label overlay for quick visual quality checks."""
    base = _robust_gray_display(gray)
    rgb = np.dstack([base, base, base])
    if labels.max() > 0:
        rgb = color.label2rgb(labels, image=rgb, alpha=alpha, bg_label=0, image_alpha=1.0)
    return np.clip(rgb, 0.0, 1.0)


def overlay_boundaries(gray: np.ndarray, labels: np.ndarray) -> np.ndarray:
    base = _robust_gray_display(gray)
    rgb = np.dstack([base, base, base])
    ov = segmentation.mark_boundaries(rgb, labels > 0, color=(1.0, 0.2, 0.2), mode="thick")
    return np.clip(ov, 0.0, 1.0)


def frame_quality_metrics(
    p_nuc: np.ndarray,
    labels: np.ndarray,
    valid_mask: np.ndarray,
    params: dict,
) -> dict:
    """Metrics used by search objective. Higher 'good' metrics are better."""
    p = clamp_probs(p_nuc)
    v = valid_mask
    m = (labels > 0) & v

    p_mass_total = float(np.sum(p[v]))
    p_mass_in_mask = float(np.sum(p[m]))
    p_mass_capture = p_mass_in_mask / max(p_mass_total, 1e-8)

    high_thr = float(params["high_recall_prob_thr"])
    low_thr = float(params["low_contam_prob_thr"])

    hi = (p >= high_thr) & v
    lo = (p <= low_thr) & v

    high_recall = float(np.mean(m[hi])) if np.any(hi) else np.nan
    low_contam = float(np.mean(lo[m])) if np.any(m) else np.nan

    n_obj = int(labels.max())
    if n_obj > 0:
        props = measure.regionprops(labels)
        areas = np.array([r.area for r in props], dtype=float)
        small_frac = float(np.mean(areas < float(params["small_obj_area_cutoff"])))
        large_frac = float(np.mean(areas > float(params["large_obj_area_cutoff"])))
        area_med = float(np.median(areas))

        solid = np.array(
            [float(r.solidity) if r.solidity is not None else np.nan for r in props],
            dtype=float,
        )
        ecc = np.array(
            [float(r.eccentricity) if r.eccentricity is not None else np.nan for r in props],
            dtype=float,
        )
        circ = []
        for r in props:
            per = float(r.perimeter) if r.perimeter is not None else np.nan
            if np.isfinite(per) and per > 0:
                circ.append(float(4.0 * np.pi * r.area / (per ** 2)))
            else:
                circ.append(np.nan)
        circ = np.array(circ, dtype=float)

        low_solidity_frac = float(np.nanmean(solid < float(params["shape_low_solidity_thr"])))
        low_circularity_frac = float(np.nanmean(circ < float(params["shape_low_circularity_thr"])))
        high_ecc_frac = float(np.nanmean(ecc > float(params["shape_high_ecc_thr"])))
        solidity_median = float(np.nanmedian(solid)) if np.any(np.isfinite(solid)) else np.nan
        circularity_median = float(np.nanmedian(circ)) if np.any(np.isfinite(circ)) else np.nan
    else:
        small_frac = np.nan
        large_frac = np.nan
        area_med = np.nan
        low_solidity_frac = np.nan
        low_circularity_frac = np.nan
        high_ecc_frac = np.nan
        solidity_median = np.nan
        circularity_median = np.nan

    # Oversplitting proxy:
    # compare object count to number of connected high-confidence probability cores.
    if np.any(hi):
        core_cc = int(ndi.label(hi.astype(np.uint8))[1])
    else:
        core_cc = 0
    if core_cc > 0:
        split_ratio = float(n_obj) / float(core_cc)
        split_over_pen = max(0.0, split_ratio - 1.25)
    else:
        split_ratio = np.nan
        split_over_pen = 0.0

    return {
        "p_mass_capture": p_mass_capture,
        "high_recall": high_recall,
        "low_contam": low_contam,
        "n_obj": n_obj,
        "small_frac": small_frac,
        "large_frac": large_frac,
        "area_median": area_med,
        "low_solidity_frac": low_solidity_frac,
        "low_circularity_frac": low_circularity_frac,
        "high_ecc_frac": high_ecc_frac,
        "solidity_median": solidity_median,
        "circularity_median": circularity_median,
        "split_ratio": split_ratio,
        "split_over_pen": split_over_pen,
        "fg_fraction": float(np.mean(m)),
    }


In [ ]:
# Quick visual sanity check before any long searches.
sanity_rows = []
n_show = min(len(PREVIEW_TIMES), 9)
fig, axes = plt.subplots(n_show, 5, figsize=(17, 2.7 * n_show), constrained_layout=True)
if n_show == 1:
    axes = axes[None, :]

for r, t in enumerate(PREVIEW_TIMES[:n_show]):
    bf, rfp, p_nuc, p_non = get_raw_prob_frame(int(t))
    seg = segment_probability_frame(p_nuc=p_nuc, valid_mask=valid_mask, params=DEFAULT_SEG_PARAMS)
    qm = frame_quality_metrics(p_nuc=p_nuc, labels=seg["labels"], valid_mask=valid_mask, params=DEFAULT_SEG_PARAMS)
    qm["time"] = int(t)
    qm["hours"] = float(t * MIN_PER_FRAME / 60.0)
    sanity_rows.append(qm)

    ax = axes[r]
    ax[0].imshow(rfp, cmap="gray")
    ax[0].set_title(f"RFP raw\nt={t}")
    ax[1].imshow(p_nuc, cmap="magma", vmin=0, vmax=1)
    ax[1].set_title("p_nuc")
    ax[2].imshow(seg["binary"], cmap="gray")
    ax[2].set_title("binary support")
    ax[3].imshow(overlay_labels_on_gray(rfp, seg["labels"], alpha=0.45))
    ax[3].set_title(f"labels alpha\nn={int(seg['labels'].max())}")
    ax[4].imshow(overlay_boundaries(rfp, seg["labels"]))
    ax[4].set_title("boundary overlay")
    for a in ax:
        a.set_xticks([])
        a.set_yticks([])
plt.show()

# Step-by-step segmentation diagnostics for 3 representative frames.
default_step_times = [0, max(0, N_TIME // 2), max(0, N_TIME - 1)]
raw_step_times = STEP_DIAG_TIMES if STEP_DIAG_TIMES is not None else default_step_times
step_diag_times = sorted({int(np.clip(t, 0, N_TIME - 1)) for t in raw_step_times})
while len(step_diag_times) < 3:
    for t in default_step_times:
        ti = int(np.clip(t, 0, N_TIME - 1))
        if ti not in step_diag_times:
            step_diag_times.append(ti)
        if len(step_diag_times) >= 3:
            break
step_diag_times = step_diag_times[:3]

step_diag_rows = []
for t in step_diag_times:
    bf, rfp, p_nuc, p_non = get_raw_prob_frame(int(t))
    seg_dbg = segment_probability_frame(
        p_nuc=p_nuc,
        valid_mask=valid_mask,
        params=DEFAULT_SEG_PARAMS,
        return_debug=True,
    )
    dbg = seg_dbg["debug_steps"]

    labels_ws = dbg["labels_watershed"]
    labels_sm = dbg["labels_splitmerge"]
    labels_final = dbg["labels_final"]
    markers = dbg["markers_final"]

    fig, axes = plt.subplots(2, 6, figsize=(22, 7), constrained_layout=True)
    ax = axes.ravel()

    ax[0].imshow(_robust_gray_display(rfp), cmap="gray", vmin=0, vmax=1)
    ax[0].set_title(f"RFP raw (display-scaled)\nt={t}")
    ax[1].imshow(dbg["p_raw"], cmap="magma", vmin=0, vmax=1)
    ax[1].set_title("1) p_nuc raw")
    ax[2].imshow(dbg["p_smooth"], cmap="magma", vmin=0, vmax=1)
    ax[2].set_title("1) p_nuc smoothed")
    ax[3].imshow(dbg["mask_low"], cmap="gray")
    ax[3].set_title(f"2) low mask\nthr={DEFAULT_SEG_PARAMS['low_prob_thr']:.2f}")
    ax[4].imshow(dbg["mask_high"], cmap="gray")
    ax[4].set_title(f"2) high mask\nthr={DEFAULT_SEG_PARAMS['high_prob_thr']:.2f}")
    ax[5].imshow(dbg["hys_pre_fallback"], cmap="gray")
    ax[5].set_title("2) hysteresis support")

    ax[6].imshow(dbg["hys_post_morph"], cmap="gray")
    ax[6].set_title("3) post-morph support")
    ax[7].imshow(markers, cmap="nipy_spectral")
    ax[7].set_title(f"4) watershed seeds\nn={int(markers.max())}")
    ax[8].imshow(overlay_boundaries(rfp, labels_ws))
    ax[8].set_title(f"4) watershed labels\nn={int(labels_ws.max())}")
    ax[9].imshow(overlay_boundaries(rfp, labels_sm))
    ax[9].set_title(f"5) after split-control\nn={int(labels_sm.max())}")
    ax[10].imshow(overlay_boundaries(rfp, labels_final))
    ax[10].set_title(f"6) final filtered labels\nn={int(labels_final.max())}")
    ax[11].imshow(overlay_labels_on_gray(rfp, labels_final, alpha=0.50))
    ax[11].set_title("Final labels alpha")

    for a in ax:
        a.set_xticks([])
        a.set_yticks([])

    fig.suptitle(f"Step-by-step segmentation diagnostics: {POSITION_LABEL}, t={t}", fontsize=14)
    out_fig = QUICK_DIR / f"stepwise_segmentation_t{int(t):03d}.png"
    fig.savefig(out_fig, dpi=170, bbox_inches="tight")
    plt.show()

    step_diag_rows.append(
        {
            "time": int(t),
            "hours": float(t * MIN_PER_FRAME / 60.0),
            "n_seed_markers": int(markers.max()),
            "n_watershed_labels": int(labels_ws.max()),
            "n_after_splitmerge": int(labels_sm.max()),
            "n_final_labels": int(labels_final.max()),
            "low_mask_fraction": float(np.mean(dbg["mask_low"])),
            "high_mask_fraction": float(np.mean(dbg["mask_high"])),
            "hys_fraction": float(np.mean(dbg["hys_pre_fallback"])),
            "postmorph_fraction": float(np.mean(dbg["hys_post_morph"])),
        }
    )

step_diag_df = pd.DataFrame(step_diag_rows).sort_values("time").reset_index(drop=True)
display(step_diag_df)
step_diag_df.to_csv(QUICK_DIR / "stepwise_segmentation_summary.csv", index=False)

sanity_df = pd.DataFrame(sanity_rows)
display(sanity_df)

# Save this first sanity table for quick audit in results dir.
sanity_df.to_csv(QUICK_DIR / "sanity_preview_metrics.csv", index=False)


In [ ]:
# Plot quick trends from sanity preview frames.
if not sanity_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), constrained_layout=True)
    axes[0].plot(sanity_df["hours"], sanity_df["n_obj"], marker="o")
    axes[0].set_title("Preview: object count")
    axes[0].set_xlabel("hours")
    axes[0].set_ylabel("n objects")

    axes[1].plot(sanity_df["hours"], sanity_df["p_mass_capture"], marker="o", label="p_mass_capture")
    axes[1].plot(sanity_df["hours"], sanity_df["high_recall"], marker="o", label="high_recall")
    axes[1].set_title("Preview: probability capture/recall")
    axes[1].set_xlabel("hours")
    axes[1].legend()

    axes[2].plot(sanity_df["hours"], sanity_df["low_contam"], marker="o", label="low_contam")
    axes[2].plot(sanity_df["hours"], sanity_df["small_frac"], marker="o", label="small_frac")
    axes[2].set_title("Preview: contamination/fragments")
    axes[2].set_xlabel("hours")
    axes[2].legend()
    plt.show()


## 4) Tracking (Division-Aware) And Temporal Metrics

Tracking is built on per-frame detections from segmentation labels.
The model supports:

- `1 -> 1` continuation
- `1 -> 2` candidate division events (later confirmed/rejected)
- births/deaths
- small gap closing


In [ ]:
TRACK_PARAMS = {
    "max_link_dist": 20.0,
    "max_area_ratio": 2.2,
    "min_area_ratio": 0.45,
    "link_cost_max": 2.1,
    "max_gap": 1,  # allow one missing frame for continuity
    # Division candidate gates (at t -> t+1 style local transitions).
    "max_division_dist": 22.0,
    "min_daughter_area_frac": 0.20,
    "max_daughter_area_frac": 0.90,
    "division_sum_area_min": 0.65,
    "division_sum_area_max": 1.75,
    "division_min_mean_prob": 0.40,
    "division_confirm_min_len": STAGE2_PERSISTENCE_K,
}


def detections_from_labels(labels: np.ndarray, p_nuc: np.ndarray, rfp: np.ndarray, time_idx: int) -> pd.DataFrame:
    """Extract per-object detection features used for linking and QC."""
    props = measure.regionprops(labels, intensity_image=p_nuc.astype(np.float32))
    rows = []
    for r in props:
        rr, cc = r.coords[:, 0], r.coords[:, 1]
        rows.append(
            {
                "time": int(time_idx),
                "label": int(r.label),
                "cy": float(r.centroid[0]),
                "cx": float(r.centroid[1]),
                "area": float(r.area),
                "mean_prob": float(r.mean_intensity),
                "max_prob": float(r.max_intensity),
                "mean_rfp": float(np.mean(rfp[rr, cc])) if rr.size > 0 else np.nan,
            }
        )
    return pd.DataFrame(rows)


@dataclass
class TrackState:
    track_id: int
    parent_id: int | None
    start_time: int
    end_time: int
    frames: list
    cy: list
    cx: list
    area: list
    mean_prob: list
    mean_rfp: list
    division_candidate: bool = False
    division_confirmed: bool = False

    def add(self, t: int, y: float, x: float, area: float, mean_prob: float, mean_rfp: float):
        self.frames.append(int(t))
        self.cy.append(float(y))
        self.cx.append(float(x))
        self.area.append(float(area))
        self.mean_prob.append(float(mean_prob))
        self.mean_rfp.append(float(mean_rfp))
        self.end_time = int(t)


def _pairwise_link_cost(prev_df: pd.DataFrame, cur_df: pd.DataFrame, tp: dict) -> np.ndarray:
    """Build cost matrix for 1->1 links with geometric + area consistency."""
    if prev_df.empty or cur_df.empty:
        return np.empty((len(prev_df), len(cur_df)), dtype=np.float32)

    py = prev_df["cy"].to_numpy(dtype=float)[:, None]
    px = prev_df["cx"].to_numpy(dtype=float)[:, None]
    pa = prev_df["area"].to_numpy(dtype=float)[:, None]

    cy = cur_df["cy"].to_numpy(dtype=float)[None, :]
    cx = cur_df["cx"].to_numpy(dtype=float)[None, :]
    ca = cur_df["area"].to_numpy(dtype=float)[None, :]

    dist = np.sqrt((py - cy) ** 2 + (px - cx) ** 2)
    area_ratio = np.maximum(ca / np.maximum(pa, 1e-6), pa / np.maximum(ca, 1e-6))

    # Composite link cost (smaller is better).
    cost = (dist / float(tp["max_link_dist"])) + 0.35 * np.abs(np.log(np.maximum(area_ratio, 1e-6)))

    # Hard gate invalid links with inf.
    bad = (dist > float(tp["max_link_dist"])) | (area_ratio > float(tp["max_area_ratio"])) | (
        1.0 / np.maximum(area_ratio, 1e-6) < float(tp["min_area_ratio"])
    )
    cost[bad] = np.inf
    return cost.astype(np.float32)


def link_tracks_division_aware(
    det_by_time: dict[int, pd.DataFrame],
    tp: dict,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Link detections across time with candidate division events and confirmation flags."""
    times = sorted(det_by_time.keys())
    tracks: dict[int, TrackState] = {}
    active_ids: list[int] = []
    next_id = 1
    detection_rows = []
    division_rows = []

    def new_track_from_det(det_row: pd.Series, parent_id: int | None = None, division_candidate: bool = False):
        nonlocal next_id
        tid = next_id
        next_id += 1
        tr = TrackState(
            track_id=tid,
            parent_id=parent_id,
            start_time=int(det_row.time),
            end_time=int(det_row.time),
            frames=[int(det_row.time)],
            cy=[float(det_row.cy)],
            cx=[float(det_row.cx)],
            area=[float(det_row.area)],
            mean_prob=[float(det_row.mean_prob)],
            mean_rfp=[float(det_row.mean_rfp)],
            division_candidate=division_candidate,
        )
        tracks[tid] = tr
        active_ids.append(tid)
        return tid

    # Initialize with first frame detections.
    t0 = times[0]
    d0 = det_by_time[t0].copy().reset_index(drop=True)
    if not d0.empty:
        for _, det in d0.iterrows():
            tid = new_track_from_det(det)
            detection_rows.append(
                {
                    "time": int(det.time),
                    "label": int(det.label),
                    "track_id": int(tid),
                    "cy": float(det.cy),
                    "cx": float(det.cx),
                    "area": float(det.area),
                    "mean_prob": float(det.mean_prob),
                    "mean_rfp": float(det.mean_rfp),
                }
            )

    # Main linking loop.
    for t in times[1:]:
        cur = det_by_time[t].copy().reset_index(drop=True)

        # Candidate active tracks that are recent enough.
        linkable_ids = [tid for tid in active_ids if (t - tracks[tid].end_time) <= int(tp["max_gap"]) + 1]
        prev_rows = []
        for tid in linkable_ids:
            tr = tracks[tid]
            prev_rows.append(
                {
                    "track_id": tid,
                    "time": tr.end_time,
                    "cy": tr.cy[-1],
                    "cx": tr.cx[-1],
                    "area": tr.area[-1],
                }
            )
        prev_df = pd.DataFrame(prev_rows)

        matched_track_ids = set()
        matched_det_idx = set()

        # 1->1 assignment via Hungarian solver.
        if not prev_df.empty and not cur.empty:
            cost = _pairwise_link_cost(prev_df, cur, tp)
            finite = np.isfinite(cost)
            if np.any(finite):
                cost_h = np.where(finite, cost, 1e6)
                rr, cc = linear_sum_assignment(cost_h)
                for i, j in zip(rr.tolist(), cc.tolist()):
                    c = float(cost_h[i, j])
                    if c <= float(tp["link_cost_max"]) and np.isfinite(cost[i, j]):
                        tid = int(prev_df.iloc[i]["track_id"])
                        det = cur.iloc[j]
                        tracks[tid].add(
                            t=int(det.time),
                            y=float(det.cy),
                            x=float(det.cx),
                            area=float(det.area),
                            mean_prob=float(det.mean_prob),
                            mean_rfp=float(det.mean_rfp),
                        )
                        matched_track_ids.add(tid)
                        matched_det_idx.add(int(j))
                        detection_rows.append(
                            {
                                "time": int(det.time),
                                "label": int(det.label),
                                "track_id": int(tid),
                                "cy": float(det.cy),
                                "cx": float(det.cx),
                                "area": float(det.area),
                                "mean_prob": float(det.mean_prob),
                                "mean_rfp": float(det.mean_rfp),
                            }
                        )

        # Candidate division handling for unmatched tracks/detections.
        unmatched_tracks = [tid for tid in linkable_ids if tid not in matched_track_ids and tracks[tid].end_time == (t - 1)]
        unmatched_det_idx = [j for j in range(len(cur)) if j not in matched_det_idx]

        if unmatched_tracks and len(unmatched_det_idx) >= 2:
            # Greedy parent-wise pairing, highest-area parents first.
            unmatched_tracks = sorted(unmatched_tracks, key=lambda z: tracks[z].area[-1], reverse=True)
            free = set(unmatched_det_idx)
            for parent_id in unmatched_tracks:
                if len(free) < 2:
                    break
                tr = tracks[parent_id]
                py, px, pa = tr.cy[-1], tr.cx[-1], tr.area[-1]
                candidates = []
                free_list = sorted(list(free))
                for i_idx in range(len(free_list)):
                    i = free_list[i_idx]
                    di = cur.iloc[i]
                    d1 = math.hypot(float(di.cy - py), float(di.cx - px))
                    if d1 > float(tp["max_division_dist"]):
                        continue
                    for j_idx in range(i_idx + 1, len(free_list)):
                        j = free_list[j_idx]
                        dj = cur.iloc[j]
                        d2 = math.hypot(float(dj.cy - py), float(dj.cx - px))
                        if d2 > float(tp["max_division_dist"]):
                            continue
                        a1, a2 = float(di.area), float(dj.area)
                        sum_ratio = (a1 + a2) / max(pa, 1e-6)
                        if not (float(tp["division_sum_area_min"]) <= sum_ratio <= float(tp["division_sum_area_max"])):
                            continue
                        # each daughter should be meaningfully sized relative to parent
                        f1 = a1 / max(pa, 1e-6)
                        f2 = a2 / max(pa, 1e-6)
                        if not (float(tp["min_daughter_area_frac"]) <= f1 <= float(tp["max_daughter_area_frac"])):
                            continue
                        if not (float(tp["min_daughter_area_frac"]) <= f2 <= float(tp["max_daughter_area_frac"])):
                            continue
                        if float(di.mean_prob) < float(tp["division_min_mean_prob"]):
                            continue
                        if float(dj.mean_prob) < float(tp["division_min_mean_prob"]):
                            continue

                        pair_cost = d1 + d2 + abs(sum_ratio - 1.0) * 8.0
                        candidates.append((pair_cost, i, j))

                if not candidates:
                    continue
                candidates.sort(key=lambda x: x[0])
                _, i_best, j_best = candidates[0]

                det_i = cur.iloc[i_best]
                det_j = cur.iloc[j_best]
                tid1 = new_track_from_det(det_i, parent_id=parent_id, division_candidate=True)
                tid2 = new_track_from_det(det_j, parent_id=parent_id, division_candidate=True)
                tracks[tid1].division_candidate = True
                tracks[tid2].division_candidate = True

                detection_rows.append(
                    {
                        "time": int(det_i.time),
                        "label": int(det_i.label),
                        "track_id": int(tid1),
                        "cy": float(det_i.cy),
                        "cx": float(det_i.cx),
                        "area": float(det_i.area),
                        "mean_prob": float(det_i.mean_prob),
                        "mean_rfp": float(det_i.mean_rfp),
                    }
                )
                detection_rows.append(
                    {
                        "time": int(det_j.time),
                        "label": int(det_j.label),
                        "track_id": int(tid2),
                        "cy": float(det_j.cy),
                        "cx": float(det_j.cx),
                        "area": float(det_j.area),
                        "mean_prob": float(det_j.mean_prob),
                        "mean_rfp": float(det_j.mean_rfp),
                    }
                )

                division_rows.append(
                    {
                        "division_time": int(t),
                        "parent_track_id": int(parent_id),
                        "daughter_track_id_1": int(tid1),
                        "daughter_track_id_2": int(tid2),
                        "parent_area_last": float(pa),
                        "daughter_area_sum": float(det_i.area + det_j.area),
                    }
                )

                free.remove(i_best)
                free.remove(j_best)

        # Any remaining unmatched detections become new births.
        unmatched_det_idx = [j for j in range(len(cur)) if j not in set([row["label"] for row in []])]
        # Recompute using already assigned detection rows at this t.
        assigned_now = set()
        for dr in detection_rows:
            if dr["time"] == int(t):
                assigned_now.add(int(dr["label"]))

        for _, det in cur.iterrows():
            if int(det.label) in assigned_now:
                continue
            tid = new_track_from_det(det, parent_id=None, division_candidate=False)
            detection_rows.append(
                {
                    "time": int(det.time),
                    "label": int(det.label),
                    "track_id": int(tid),
                    "cy": float(det.cy),
                    "cx": float(det.cx),
                    "area": float(det.area),
                    "mean_prob": float(det.mean_prob),
                    "mean_rfp": float(det.mean_rfp),
                }
            )

        # Retire stale tracks.
        active_ids = [tid for tid in active_ids if (t - tracks[tid].end_time) <= int(tp["max_gap"]) + 1]

    det_track_df = pd.DataFrame(detection_rows).sort_values(["time", "track_id"]).reset_index(drop=True)

    # Track summary.
    track_rows = []
    for tid, tr in tracks.items():
        track_rows.append(
            {
                "track_id": int(tid),
                "parent_id": int(tr.parent_id) if tr.parent_id is not None else np.nan,
                "start_time": int(tr.start_time),
                "end_time": int(tr.end_time),
                "length": int(len(tr.frames)),
                "mean_area": float(np.mean(tr.area)),
                "mean_prob": float(np.mean(tr.mean_prob)),
                "max_prob": float(np.max(tr.mean_prob)),
                "division_candidate_track": bool(tr.division_candidate),
            }
        )
    track_df = pd.DataFrame(track_rows).sort_values("track_id").reset_index(drop=True)

    div_df = pd.DataFrame(division_rows)
    if not div_df.empty:
        lens = track_df.set_index("track_id")["length"].to_dict()
        confirmed = []
        for row in div_df.itertuples(index=False):
            l1 = int(lens.get(int(row.daughter_track_id_1), 0))
            l2 = int(lens.get(int(row.daughter_track_id_2), 0))
            ok = (l1 >= int(tp["division_confirm_min_len"])) and (l2 >= int(tp["division_confirm_min_len"]))
            confirmed.append(bool(ok))
        div_df["confirmed"] = confirmed
    else:
        div_df["confirmed"] = pd.Series(dtype=bool)

    return det_track_df, track_df, div_df


In [ ]:
# Run a quick tracking sanity pass on a contiguous window.
# Tracking needs adjacent frames; sparse time sampling inflates one-frame tracks.
quick_window_len = min(60, N_TIME)
quick_start = max(0, (N_TIME // 2) - (quick_window_len // 2))
quick_track_times = np.arange(quick_start, quick_start + quick_window_len, dtype=int)

det_by_t_quick = {}
seg_cache_quick = {}
frame_metric_rows = []

t0 = time.time()
for t in quick_track_times:
    bf, rfp, p_nuc, p_non = get_raw_prob_frame(int(t))
    seg = segment_probability_frame(p_nuc=p_nuc, valid_mask=valid_mask, params=DEFAULT_SEG_PARAMS)
    det = detections_from_labels(seg["labels"], p_nuc, rfp, time_idx=int(t))
    det_by_t_quick[int(t)] = det
    seg_cache_quick[int(t)] = seg

    qm = frame_quality_metrics(p_nuc=p_nuc, labels=seg["labels"], valid_mask=valid_mask, params=DEFAULT_SEG_PARAMS)
    qm.update({"time": int(t), "hours": float(t * MIN_PER_FRAME / 60.0)})
    frame_metric_rows.append(qm)

quick_det_track_df, quick_track_df, quick_div_df = link_tracks_division_aware(det_by_t_quick, TRACK_PARAMS)
dt = time.time() - t0

quick_frame_metrics_df = pd.DataFrame(frame_metric_rows).sort_values("time").reset_index(drop=True)

print(f"Quick pass runtime: {dt:.2f}s over {len(quick_track_times)} frames")
print(f"Tracks: {len(quick_track_df)}")
print(f"Division candidates: {len(quick_div_df)}")
if len(quick_div_df) > 0:
    print(f"Division confirmed: {int(quick_div_df['confirmed'].sum())} / {len(quick_div_df)}")

display(quick_frame_metrics_df.head())
display(quick_track_df.head())
display(quick_div_df.head())


In [ ]:
# Temporal diagnostics from quick pass.
if not quick_frame_metrics_df.empty:
    count_by_t = quick_frame_metrics_df[["time", "n_obj"]].copy().sort_values("time")
    counts = count_by_t["n_obj"].to_numpy(dtype=float)
    if len(counts) >= 6:
        early = float(np.median(counts[: max(2, len(counts) // 8)]))
        late = float(np.median(counts[-max(2, len(counts) // 8) :]))
        growth = (late - early) / max(early, 1.0)
    else:
        growth = np.nan
    print(f"Quick count growth score (early->late): {growth:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

axes[0].plot(quick_frame_metrics_df["hours"], quick_frame_metrics_df["n_obj"], marker="o")
axes[0].set_title("Quick pass: object count")
axes[0].set_xlabel("hours")
axes[0].set_ylabel("n objects")

axes[1].plot(quick_frame_metrics_df["hours"], quick_frame_metrics_df["p_mass_capture"], marker="o", label="p_mass_capture")
axes[1].plot(quick_frame_metrics_df["hours"], quick_frame_metrics_df["high_recall"], marker="o", label="high_recall")
axes[1].set_title("Quick pass: recall/capture")
axes[1].set_xlabel("hours")
axes[1].legend()

if not quick_track_df.empty:
    axes[2].hist(quick_track_df["length"], bins=20, color="#4C78A8", alpha=0.9)
axes[2].set_title("Track length distribution (quick)")
axes[2].set_xlabel("track length (frames)")
axes[2].set_ylabel("count")
plt.show()


## 5) Candidate Search With Progressive Runtime

Stage 1 uses sampled frames and framewise metrics only.
Stage 2 (optional) uses richer temporal + division metrics on larger contiguous windows.

This is designed to avoid expensive long searches until quick checks look correct.


In [ ]:
# Candidate space for stage-1 random search.
STAGE1_SPACE = {
    "smooth_sigma": [0.8, 1.0, 1.2, 1.4],
    "low_prob_thr": [0.24, 0.28, 0.30, 0.33, 0.36],
    "high_prob_thr": [0.48, 0.52, 0.56, 0.60, 0.64],
    "seed_prob_thr": [0.34, 0.38, 0.42, 0.46, 0.50],
    "seed_min_distance": [5, 6, 7, 8],
    "min_area": [45, 60, 75, 90],
    "max_area": [1400, 1800, 2200, 2800],
    "min_solidity": [0.45, 0.50, 0.55, 0.60],
    "min_mean_prob": [0.30, 0.34, 0.38, 0.42],
}


def sample_candidate_dicts(space: dict, n: int, seed: int) -> list[dict]:
    """Randomly sample unique candidate dictionaries from a discrete parameter space."""
    rng = np.random.default_rng(seed)
    keys = list(space.keys())
    all_vals = [space[k] for k in keys]
    # If space is small enough, enumerate and sample unique tuples.
    total = int(np.prod([len(v) for v in all_vals], dtype=np.int64))
    if total <= 200000:
        combos = list(product(*all_vals))
        choose = rng.choice(len(combos), size=min(n, len(combos)), replace=False)
        out = []
        for idx in choose:
            vals = combos[int(idx)]
            d = {k: vals[i] for i, k in enumerate(keys)}
            out.append(d)
        return out

    # Fallback random draws for huge spaces.
    out = []
    seen = set()
    while len(out) < n:
        vals = tuple(space[k][int(rng.integers(0, len(space[k])))] for k in keys)
        if vals in seen:
            continue
        seen.add(vals)
        out.append({k: vals[i] for i, k in enumerate(keys)})
    return out


def merge_params(base: dict, update: dict) -> dict:
    d = dict(base)
    d.update(update)
    # Keep thresholds ordered.
    if d["high_prob_thr"] < d["low_prob_thr"] + 0.08:
        d["high_prob_thr"] = float(min(0.90, d["low_prob_thr"] + 0.08))
    if d["seed_prob_thr"] < d["low_prob_thr"] + 0.03:
        d["seed_prob_thr"] = float(min(0.90, d["low_prob_thr"] + 0.03))
    # Integer casts for integer params.
    for k in ["seed_min_distance", "min_area", "max_area"]:
        d[k] = int(round(d[k]))
    return d


def stratified_time_sample(n_time: int, n_pick: int, seed: int) -> np.ndarray:
    """Sample frames across early/mid/late periods for robust quick-stage checks."""
    rng = np.random.default_rng(seed)
    bins = np.array_split(np.arange(n_time, dtype=int), 8)
    per_bin = max(1, int(np.ceil(n_pick / len(bins))))
    picks = []
    for b in bins:
        if b.size == 0:
            continue
        size = min(per_bin, b.size)
        choose = rng.choice(b, size=size, replace=False)
        picks.extend(choose.tolist())
    picks = sorted(set(picks))
    if len(picks) > n_pick:
        picks = sorted(rng.choice(np.array(picks, dtype=int), size=n_pick, replace=False).tolist())
    return np.array(picks, dtype=int)


def stage1_objective(metrics_df: pd.DataFrame) -> float:
    """Framewise objective. Higher is better."""
    if metrics_df.empty:
        return -np.inf
    m = metrics_df.copy()

    def _med(col, fallback=0.0):
        x = m[col].dropna()
        return float(np.median(x)) if len(x) else float(fallback)

    p_cap = _med("p_mass_capture")
    recall = _med("high_recall")
    contam = _med("low_contam")
    smallf = _med("small_frac")
    largef = _med("large_frac")
    fgf = _med("fg_fraction")
    low_sol_f = _med("low_solidity_frac")
    low_circ_f = _med("low_circularity_frac")
    high_ecc_f = _med("high_ecc_frac")
    split_pen = _med("split_over_pen")

    counts = m.sort_values("time")["n_obj"].to_numpy(dtype=float)
    if len(counts) >= 10:
        early = float(np.median(counts[: max(3, len(counts) // 10)]))
        late = float(np.median(counts[-max(3, len(counts) // 10) :]))
        growth = (late - early) / max(early, 1.0)
        # Abrupt negative cliffs penalty.
        smooth = pd.Series(counts).rolling(5, center=True, min_periods=1).median().to_numpy()
        dif = np.diff(smooth)
        cliff = float(np.min(dif)) / max(np.median(smooth), 1.0)
        cliff_pen = max(0.0, -cliff)
    else:
        growth = 0.0
        cliff_pen = 0.0

    score = (
        2.2 * p_cap
        + 2.0 * recall
        - 1.5 * contam
        - 0.9 * smallf
        - 0.8 * largef
        - 2.2 * low_sol_f
        - 2.0 * low_circ_f
        - 1.2 * high_ecc_f
        - 1.4 * split_pen
        + 0.5 * growth
        - 0.8 * cliff_pen
        + 0.2 * fgf
    )
    return float(score)


In [ ]:
def evaluate_candidate_stage1(candidate_params: dict, times: np.ndarray) -> tuple[float, pd.DataFrame]:
    rows = []
    for t in times:
        bf, rfp, p_nuc, p_non = get_raw_prob_frame(int(t))
        seg = segment_probability_frame(p_nuc=p_nuc, valid_mask=valid_mask, params=candidate_params)
        qm = frame_quality_metrics(p_nuc=p_nuc, labels=seg["labels"], valid_mask=valid_mask, params=candidate_params)
        qm["time"] = int(t)
        rows.append(qm)
    df = pd.DataFrame(rows)
    score = stage1_objective(df)
    return float(score), df


# Runtime pre-check so long runs are predictable.
bench_times = np.linspace(0, N_TIME - 1, 16, dtype=int)
bench_times = np.unique(bench_times)
t0 = time.time()
for t in bench_times:
    _, _, p_nuc, _ = get_raw_prob_frame(int(t))
    _ = segment_probability_frame(p_nuc=p_nuc, valid_mask=valid_mask, params=DEFAULT_SEG_PARAMS)
per_frame_sec = (time.time() - t0) / max(len(bench_times), 1)
print(f"Benchmark segmentation runtime: {per_frame_sec:.4f} sec/frame")

stage1_times = stratified_time_sample(N_TIME, STAGE1_SAMPLE_N_FRAMES, STAGE1_RANDOM_SEED)
est_stage1_sec = per_frame_sec * len(stage1_times) * STAGE1_N_CANDIDATES
print(f"Stage-1 sample frames: {len(stage1_times)}")
print(f"Estimated stage-1 runtime: ~{est_stage1_sec / 60.0:.1f} minutes")


In [ ]:
stage1_results = []
stage1_top_metrics = {}

if RUN_STAGE1_SEARCH:
    cand_updates = sample_candidate_dicts(STAGE1_SPACE, STAGE1_N_CANDIDATES, STAGE1_RANDOM_SEED)
    t0 = time.time()
    for i, upd in enumerate(cand_updates, start=1):
        cand = merge_params(DEFAULT_SEG_PARAMS, upd)
        score, mdf = evaluate_candidate_stage1(cand, stage1_times)
        stage1_results.append(
            {
                "candidate_id": int(i),
                "score_stage1": float(score),
                **{k: cand[k] for k in STAGE1_SPACE.keys()},
            }
        )

        # Keep metrics for a small set of best candidates for later visual inspection.
        if (i <= 5) or (len(stage1_top_metrics) < 10):
            stage1_top_metrics[i] = mdf

        if (i % 8 == 0) or (i == len(cand_updates)):
            elapsed = time.time() - t0
            print(f"[stage1] {i:3d}/{len(cand_updates)} done | elapsed {elapsed/60.0:.2f} min")

    stage1_df = pd.DataFrame(stage1_results).sort_values("score_stage1", ascending=False).reset_index(drop=True)
    display(stage1_df.head(15))
    stage1_df.to_csv(SEARCH_DIR / "stage1_scores.csv", index=False)
    print(f"Saved: {SEARCH_DIR / 'stage1_scores.csv'}")
else:
    stage1_df = pd.DataFrame()
    print("Stage-1 search skipped by config.")


In [ ]:
# Visual comparison of top stage-1 candidates on sentinel frames.
if not stage1_df.empty:
    topN = min(4, len(stage1_df))
    sentinel_times = [0, 40, 120, 200, 299]
    sentinel_times = [t for t in sentinel_times if 0 <= t < N_TIME]

    fig, axes = plt.subplots(
        len(sentinel_times),
        topN + 1,
        figsize=(3.4 * (topN + 1), 2.8 * len(sentinel_times)),
        constrained_layout=True,
    )
    if len(sentinel_times) == 1:
        axes = axes[None, :]

    for r, t in enumerate(sentinel_times):
        _, rfp, p_nuc, _ = get_raw_prob_frame(int(t))
        axes[r, 0].imshow(rfp, cmap="gray")
        axes[r, 0].set_title(f"RFP raw\nt={t}")
        axes[r, 0].set_xticks([])
        axes[r, 0].set_yticks([])

        for c in range(topN):
            row = stage1_df.iloc[c]
            upd = {k: row[k] for k in STAGE1_SPACE.keys()}
            params_c = merge_params(DEFAULT_SEG_PARAMS, upd)
            seg = segment_probability_frame(p_nuc, valid_mask, params_c)
            axes[r, c + 1].imshow(overlay_boundaries(rfp, seg["labels"]))
            axes[r, c + 1].set_title(
                f"cand#{int(row.candidate_id)} score={row.score_stage1:.3f}\nn={int(seg['labels'].max())}"
            )
            axes[r, c + 1].set_xticks([])
            axes[r, c + 1].set_yticks([])
    plt.show()


In [ ]:
# ---------- Stage 2 objective with temporal + division metrics ----------
def run_segmentation_over_times(params: dict, times: np.ndarray):
    seg_by_t = {}
    det_by_t = {}
    fm_rows = []
    for t in times:
        bf, rfp, p_nuc, p_non = get_raw_prob_frame(int(t))
        seg = segment_probability_frame(p_nuc, valid_mask, params)
        det = detections_from_labels(seg["labels"], p_nuc, rfp, int(t))
        seg_by_t[int(t)] = seg
        det_by_t[int(t)] = det
        qm = frame_quality_metrics(p_nuc, seg["labels"], valid_mask, params)
        qm.update({"time": int(t)})
        fm_rows.append(qm)
    fm_df = pd.DataFrame(fm_rows).sort_values("time").reset_index(drop=True)
    det_track_df, track_df, div_df = link_tracks_division_aware(det_by_t, TRACK_PARAMS)
    return seg_by_t, det_by_t, fm_df, det_track_df, track_df, div_df


def temporal_objective(fm_df: pd.DataFrame, track_df: pd.DataFrame, div_df: pd.DataFrame) -> dict:
    # Framewise core terms reused from stage-1.
    s1 = stage1_objective(fm_df)

    def _med(col, fallback=0.0):
        x = fm_df[col].dropna()
        return float(np.median(x)) if len(x) else float(fallback)

    if track_df.empty:
        track_len_med = 0.0
        short_frac = 1.0
        one_frame_frac = 1.0
    else:
        lens = track_df["length"].to_numpy(dtype=float)
        track_len_med = float(np.median(lens))
        short_frac = float(np.mean(lens <= 2))
        one_frame_frac = float(np.mean(lens <= 1))

    if div_df.empty:
        div_confirm_rate = 0.0
        false_split_rate = 0.0
        n_div = 0
    else:
        n_div = int(len(div_df))
        conf = div_df["confirmed"].to_numpy(dtype=bool)
        div_confirm_rate = float(np.mean(conf))
        false_split_rate = float(np.mean(~conf))

    # Count smoothness from framewise n_obj.
    counts = fm_df.sort_values("time")["n_obj"].to_numpy(dtype=float)
    if len(counts) > 5:
        smooth = pd.Series(counts).rolling(7, center=True, min_periods=1).median().to_numpy()
        d = np.diff(smooth)
        neg_jumps = d[d < 0]
        jump_pen = float(np.mean(np.abs(neg_jumps))) / max(np.median(smooth), 1.0) if len(neg_jumps) else 0.0
    else:
        jump_pen = 0.0

    # Additional shape / oversplitting penalties from framewise metrics.
    low_sol_f = _med("low_solidity_frac")
    low_circ_f = _med("low_circularity_frac")
    high_ecc_f = _med("high_ecc_frac")
    split_pen = _med("split_over_pen")

    score = (
        s1
        + 0.65 * min(track_len_med / 15.0, 1.5)
        - 0.9 * short_frac
        - 0.6 * one_frame_frac
        + 0.7 * div_confirm_rate
        - 1.0 * false_split_rate
        - 0.7 * jump_pen
        - 1.3 * low_sol_f
        - 1.2 * low_circ_f
        - 0.7 * high_ecc_f
        - 0.9 * split_pen
    )
    return {
        "score_stage1_component": float(s1),
        "track_len_median": float(track_len_med),
        "short_track_frac": float(short_frac),
        "one_frame_track_frac": float(one_frame_frac),
        "division_confirm_rate": float(div_confirm_rate),
        "false_split_rate": float(false_split_rate),
        "count_jump_penalty": float(jump_pen),
        "low_solidity_frac_med": float(low_sol_f),
        "low_circularity_frac_med": float(low_circ_f),
        "high_ecc_frac_med": float(high_ecc_f),
        "split_over_pen_med": float(split_pen),
        "n_division_candidates": int(n_div),
        "score_stage2": float(score),
    }


In [ ]:
stage2_df = pd.DataFrame()
stage2_best_params = dict(DEFAULT_SEG_PARAMS)

if RUN_STAGE2_SEARCH and (not stage1_df.empty):
    # Build contiguous evaluation windows (important for temporal metrics/tracking).
    # We use full time range for final ranking on this single-position pilot.
    if STAGE2_EVAL_MAX_FRAMES is None:
        eval_times = np.arange(N_TIME, dtype=int)
    else:
        eval_times = np.arange(min(int(STAGE2_EVAL_MAX_FRAMES), N_TIME), dtype=int)

    # Build stage-2 candidate set from top stage-1 + local jitters.
    rng = np.random.default_rng(STAGE2_RANDOM_SEED)
    top_rows = stage1_df.head(int(STAGE2_TOPK_FROM_STAGE1))
    stage2_candidates = []
    for _, row in top_rows.iterrows():
        base_upd = {k: row[k] for k in STAGE1_SPACE.keys()}
        base = merge_params(DEFAULT_SEG_PARAMS, base_upd)
        stage2_candidates.append(base)
        # local refinements around each top candidate
        for _ in range(int(STAGE2_N_REFINEMENTS_PER_BASE)):
            cand = dict(base)
            cand["low_prob_thr"] = float(np.clip(cand["low_prob_thr"] + rng.normal(0, 0.02), 0.18, 0.55))
            cand["high_prob_thr"] = float(np.clip(cand["high_prob_thr"] + rng.normal(0, 0.025), 0.30, 0.90))
            cand["seed_prob_thr"] = float(np.clip(cand["seed_prob_thr"] + rng.normal(0, 0.02), 0.20, 0.85))
            cand["smooth_sigma"] = float(np.clip(cand["smooth_sigma"] + rng.normal(0, 0.12), 0.6, 2.2))
            cand["min_area"] = int(np.clip(round(cand["min_area"] + rng.normal(0, 12)), 30, 180))
            cand["max_area"] = int(np.clip(round(cand["max_area"] + rng.normal(0, 220)), 900, 4200))
            cand["min_mean_prob"] = float(np.clip(cand["min_mean_prob"] + rng.normal(0, 0.03), 0.20, 0.60))
            cand = merge_params(DEFAULT_SEG_PARAMS, cand)
            stage2_candidates.append(cand)

    # Deduplicate candidate dictionaries.
    unique = []
    seen = set()
    for c in stage2_candidates:
        key = tuple((k, c[k]) for k in sorted(c.keys()))
        if key in seen:
            continue
        seen.add(key)
        unique.append(c)
    stage2_candidates = unique

    # Runtime estimate from one benchmark candidate.
    t0 = time.time()
    _ = run_segmentation_over_times(stage2_candidates[0], eval_times[:60])
    sec_per_frame = (time.time() - t0) / 60.0
    est_total_min = sec_per_frame * len(eval_times) * len(stage2_candidates) / 60.0
    print(f"[stage2] candidates={len(stage2_candidates)}, eval_frames={len(eval_times)}")
    print(f"[stage2] estimated runtime ~{est_total_min:.1f} min (rough)")

    rows = []
    t_all = time.time()
    for i, cand in enumerate(stage2_candidates, start=1):
        _, _, fm_df, det_track_df, track_df, div_df = run_segmentation_over_times(cand, eval_times)
        met = temporal_objective(fm_df, track_df, div_df)
        rows.append({"candidate_id": i, **met, **cand})

        if (i % 2 == 0) or (i == len(stage2_candidates)):
            elapsed = (time.time() - t_all) / 60.0
            print(f"[stage2] {i:3d}/{len(stage2_candidates)} done | elapsed {elapsed:.2f} min")

    stage2_df = pd.DataFrame(rows).sort_values("score_stage2", ascending=False).reset_index(drop=True)
    display(stage2_df.head(12))
    stage2_df.to_csv(SEARCH_DIR / "stage2_scores.csv", index=False)
    stage2_best_params = {k: stage2_df.iloc[0][k] for k in DEFAULT_SEG_PARAMS.keys()}
    with open(SEARCH_DIR / "best_params_stage2.json", "w") as f:
        json.dump(stage2_best_params, f, indent=2)
    print(f"Saved: {SEARCH_DIR / 'stage2_scores.csv'}")
    print(f"Saved: {SEARCH_DIR / 'best_params_stage2.json'}")
else:
    if stage1_df.empty:
        print("Stage-2 not run because stage-1 has no results.")
    else:
        print("Stage-2 skipped by config.")


In [ ]:
# Choose final params:
# - stage2 winner if stage2 ran,
# - otherwise stage1 winner,
# - otherwise defaults.
if not stage2_df.empty:
    FINAL_PARAMS = {k: stage2_df.iloc[0][k] for k in DEFAULT_SEG_PARAMS.keys()}
    final_source = "stage2_best"
elif not stage1_df.empty:
    row = stage1_df.iloc[0]
    upd = {k: row[k] for k in STAGE1_SPACE.keys()}
    FINAL_PARAMS = merge_params(DEFAULT_SEG_PARAMS, upd)
    final_source = "stage1_best"
else:
    FINAL_PARAMS = dict(DEFAULT_SEG_PARAMS)
    final_source = "defaults"

print("Final parameter source:", final_source)
print(json.dumps(FINAL_PARAMS, indent=2))
with open(SEARCH_DIR / "final_params_selected.json", "w") as f:
    json.dump(FINAL_PARAMS, f, indent=2)


## 6) Full Run With Final Params + Exports

Exports:

- Per-frame segmentation metrics CSV
- Detection-to-track table CSV
- Track summary CSV
- Division candidate/confirmation CSV
- Optional full overlay stack TIFF
- Optional time-ordered montage PNG


In [ ]:
if FULL_RUN_MAX_FRAMES is None:
    full_times = np.arange(N_TIME, dtype=int)
else:
    full_times = np.arange(min(int(FULL_RUN_MAX_FRAMES), N_TIME), dtype=int)
seg_by_t = {}
det_by_t = {}
fm_rows = []

t0 = time.time()
for i, t in enumerate(full_times, start=1):
    bf, rfp, p_nuc, p_non = get_raw_prob_frame(int(t))
    seg = segment_probability_frame(p_nuc, valid_mask, FINAL_PARAMS)
    det = detections_from_labels(seg["labels"], p_nuc, rfp, int(t))
    seg_by_t[int(t)] = seg
    det_by_t[int(t)] = det

    qm = frame_quality_metrics(p_nuc, seg["labels"], valid_mask, FINAL_PARAMS)
    qm.update({"time": int(t), "hours": float(t * MIN_PER_FRAME / 60.0)})
    fm_rows.append(qm)

    if (i % 50 == 0) or (i == len(full_times)):
        print(f"[full] segmented {i}/{len(full_times)} frames")

full_fm_df = pd.DataFrame(fm_rows).sort_values("time").reset_index(drop=True)
full_det_track_df, full_track_df, full_div_df = link_tracks_division_aware(det_by_t, TRACK_PARAMS)

full_fm_df.to_csv(EXPORT_DIR / "frame_metrics.csv", index=False)
full_det_track_df.to_csv(EXPORT_DIR / "detections_tracks.csv", index=False)
full_track_df.to_csv(EXPORT_DIR / "track_summary.csv", index=False)
full_div_df.to_csv(EXPORT_DIR / "division_events.csv", index=False)
print(f"Saved tables to {EXPORT_DIR}")
print(f"Full run runtime: {(time.time() - t0)/60.0:.2f} min")


In [ ]:
# Temporal cleanup diagnostic: one-frame tracks often represent flicker false positives.
if not full_track_df.empty:
    one_frame_ids = set(full_track_df.loc[full_track_df["length"] <= 1, "track_id"].astype(int).tolist())
    keep_det_mask = ~full_det_track_df["track_id"].astype(int).isin(one_frame_ids)
    det_clean_df = full_det_track_df[keep_det_mask].copy()
else:
    one_frame_ids = set()
    det_clean_df = full_det_track_df.copy()

print(f"One-frame tracks removed in diagnostic view: {len(one_frame_ids)}")
print(f"Remaining detections after cleanup: {len(det_clean_df)}")

# Count trajectories before/after one-frame cleanup.
raw_counts = full_det_track_df.groupby("time").size().rename("count_raw")
clean_counts = det_clean_df.groupby("time").size().rename("count_clean")
count_df = pd.concat([raw_counts, clean_counts], axis=1).fillna(0).reset_index()

fig, ax = plt.subplots(1, 1, figsize=(10, 4), constrained_layout=True)
ax.plot(count_df["time"] * (MIN_PER_FRAME / 60.0), count_df["count_raw"], label="raw linked detections", alpha=0.8)
ax.plot(count_df["time"] * (MIN_PER_FRAME / 60.0), count_df["count_clean"], label="cleaned (drop one-frame tracks)", alpha=0.9)
ax.set_title("Linked detection counts over time")
ax.set_xlabel("hours")
ax.set_ylabel("count")
ax.legend()
plt.show()


In [ ]:
# Build per-frame overlays from final labels.
overlays_uint8 = []
overlay_index_rows = []

for t in full_times:
    bf, rfp, p_nuc, p_non = get_raw_prob_frame(int(t))
    labels = seg_by_t[int(t)]["labels"]
    ov = overlay_boundaries(rfp, labels)  # float rgb in [0,1]
    ov8 = np.clip(np.round(ov * 255.0), 0, 255).astype(np.uint8)
    overlays_uint8.append(ov8)
    overlay_index_rows.append(
        {
            "position_label": POSITION_LABEL,
            "time": int(t),
            "hours": float(t * MIN_PER_FRAME / 60.0),
            "n_labels": int(labels.max()),
        }
    )

overlay_index_df = pd.DataFrame(overlay_index_rows)
overlay_index_df.to_csv(EXPORT_DIR / "overlay_index.csv", index=False)

if EXPORT_FULL_OVERLAY_STACK:
    stack_path = EXPORT_DIR / f"{POSITION_LABEL}_overlay_stack.tif"
    tifffile.imwrite(
        stack_path,
        np.stack(overlays_uint8, axis=0),
        photometric="rgb",
        compression=None,
    )
    print(f"Saved overlay stack: {stack_path}")

if EXPORT_MONTAGE:
    n = len(overlays_uint8)
    cols = int(MONTAGE_COLS)
    rows = int(np.ceil(n / cols))
    h, w = overlays_uint8[0].shape[:2]
    montage = np.zeros((rows * h, cols * w, 3), dtype=np.uint8)
    for i, ov in enumerate(overlays_uint8):
        rr = i // cols
        cc = i % cols
        montage[rr * h : (rr + 1) * h, cc * w : (cc + 1) * w] = ov
    montage_path = EXPORT_DIR / f"{POSITION_LABEL}_overlay_montage.png"
    tifffile.imwrite(montage_path, montage)
    print(f"Saved montage: {montage_path}")


In [ ]:
# Final QC summary panel.
summary = {}
summary["n_frames"] = int(len(full_fm_df))
summary["median_n_objects"] = float(full_fm_df["n_obj"].median()) if not full_fm_df.empty else np.nan
summary["median_p_mass_capture"] = float(full_fm_df["p_mass_capture"].median()) if not full_fm_df.empty else np.nan
summary["median_high_recall"] = float(full_fm_df["high_recall"].median()) if not full_fm_df.empty else np.nan
summary["median_low_contam"] = float(full_fm_df["low_contam"].median()) if not full_fm_df.empty else np.nan
summary["n_tracks"] = int(len(full_track_df))
summary["median_track_len"] = float(full_track_df["length"].median()) if not full_track_df.empty else np.nan
summary["one_frame_track_frac"] = float(np.mean(full_track_df["length"] <= 1)) if not full_track_df.empty else np.nan
summary["n_division_candidates"] = int(len(full_div_df))
summary["division_confirm_rate"] = float(full_div_df["confirmed"].mean()) if (not full_div_df.empty) else np.nan

summary_df = pd.DataFrame([summary])
display(summary_df)
summary_df.to_csv(EXPORT_DIR / "qc_summary.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0, 0].plot(full_fm_df["hours"], full_fm_df["n_obj"], lw=1.5)
axes[0, 0].set_title("Per-frame object count")
axes[0, 0].set_xlabel("hours")
axes[0, 0].set_ylabel("n")

axes[0, 1].plot(full_fm_df["hours"], full_fm_df["p_mass_capture"], label="p_mass_capture")
axes[0, 1].plot(full_fm_df["hours"], full_fm_df["high_recall"], label="high_recall")
axes[0, 1].set_title("Probability capture/recall")
axes[0, 1].set_xlabel("hours")
axes[0, 1].legend()

axes[1, 0].plot(full_fm_df["hours"], full_fm_df["low_contam"], label="low_contam")
axes[1, 0].plot(full_fm_df["hours"], full_fm_df["small_frac"], label="small_frac")
axes[1, 0].plot(full_fm_df["hours"], full_fm_df["large_frac"], label="large_frac")
axes[1, 0].set_title("Contamination + object-size diagnostics")
axes[1, 0].set_xlabel("hours")
axes[1, 0].legend()

if not full_track_df.empty:
    axes[1, 1].hist(full_track_df["length"], bins=30, color="#4C78A8")
axes[1, 1].set_title("Track length distribution")
axes[1, 1].set_xlabel("length (frames)")
axes[1, 1].set_ylabel("count")
plt.show()


## Notes

- This notebook intentionally separates **quick sanity** from **long optimization**.
- For robust division quantification, increase `TRACK_PARAMS["division_confirm_min_len"]` if needed.
- For dim-nucleus recall, tune (`low_prob_thr`, `seed_prob_thr`, `min_mean_prob`) first.
- For false positives, tune (`min_area`, `min_solidity`, and one-frame-track cleanup policy).
